# Step 30 — EDA (interactive)

Reuses the same functions as `run_eda.py` so this notebook and the generated `report.md` never drift apart. Use this notebook for ad hoc follow-up questions the fixed report doesn't answer — e.g. filtering to a specific source, or looking at a job family in more detail.

Requires `NEON_PROD_DATABASE_URL` to be set in the environment before starting Jupyter (never commit its value).

In [ ]:
import run_eda as eda

conn = eda.get_connection()
jobs = eda.load_jobs(conn)
salaries = eda.load_salaries(conn)
skills = eda.load_skills(conn)
completeness = eda.data_completeness(conn)
conn.dispose()

print(f"{len(jobs)} jobs loaded")
jobs.head()

## Data completeness

In [ ]:
completeness

## Title classification breakdown

In [ ]:
jobs['title_classification_status'].value_counts(normalize=True).mul(100).round(1)

In [ ]:
from IPython.display import Image

eda.chart_title_classification(jobs)
Image(eda.OUTPUT_DIR / '02_title_classification.png')

## Job families (matched only) — explore freely below

In [ ]:
matched = jobs[jobs['title_classification_status'] == 'matched']
matched['job_family'].value_counts().head(20)

In [ ]:
# Example ad hoc query: drill into a single job family's raw titles
family = 'Full-stack Engineering'
cols = ['job_title', 'normalized_title', 'company_name', 'source_name']
matched[matched['job_family'] == family][cols]

## Skills

In [ ]:
skills['skill_name'].value_counts().head(20)

## Salary (small sample — see report.md caveats)

In [ ]:
eda.salary_summary(salaries)

## Regenerate the full fixed report
Run this if you want `output/report.md` and all PNGs refreshed from this session's data without leaving the notebook.

In [ ]:
eda.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
charts = {}
charts['source_status'] = eda.chart_source_status(jobs)
charts['title_classification'] = eda.chart_title_classification(jobs)
charts['job_families'] = eda.chart_job_families(jobs)
charts['posting_volume'], outliers = eda.chart_posting_volume(jobs)
charts['top_companies'] = eda.chart_top_companies(jobs)
charts['top_skills'] = eda.chart_top_skills(skills, jobs_total=len(jobs))
salary_rows = eda.salary_summary(salaries)
report = eda.build_report(jobs, salaries, skills, completeness, charts, outliers, salary_rows)
(eda.OUTPUT_DIR / 'report.md').write_text(report, encoding='utf-8')
print('report.md regenerated')